In [87]:
import importlib
import src.writer

importlib.reload(src.writer)


<module 'src.writer' from '/Users/Kiran/Desktop/HDB-Data-Engineering-Assignment_2026/src/writer.py'>

In [32]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project Root: {PROJECT_ROOT}")

Project Root: /Users/Kiran/Desktop/HDB-Data-Engineering-Assignment_2026


In [19]:
from src.spark_session import create_spark

spark = create_spark()

spark

In [20]:
from src.download import DownloadManager

downloader = DownloadManager()

downloader.download_all()



------------------------------------------------------------
Dataset : resale_2012_2014
✓ resale_2012_2014.csv already exists
------------------------------------------------------------
Dataset : resale_2015_2016
✓ resale_2015_2016.csv already exists

All downloads completed.


In [21]:
from pathlib import Path

raw_path = PROJECT_ROOT / "data" / "raw"

for file in raw_path.glob("*.csv"):
    print(file.name)

resale_2012_2014.csv
resale_2015_2016.csv


In [22]:
from src.loader import load_master_dataset

master_df = load_master_dataset(spark)

Reading resale_2012_2014.csv


Reading resale_2015_2016.csv


In [25]:
master_df.printSchema()
master_df.show(5, truncate=False)

root
 |-- month: timestamp (nullable = true)
 |-- town: string (nullable = true)
 |-- flat_type: string (nullable = true)
 |-- block: string (nullable = true)
 |-- street_name: string (nullable = true)
 |-- storey_range: string (nullable = true)
 |-- floor_area_sqm: double (nullable = true)
 |-- flat_model: string (nullable = true)
 |-- lease_commence_date: integer (nullable = true)
 |-- resale_price: double (nullable = true)
 |-- remaining_lease: integer (nullable = true)

+-------------------+----------+---------+-----+-----------------+------------+--------------+--------------+-------------------+------------+---------------+
|month              |town      |flat_type|block|street_name      |storey_range|floor_area_sqm|flat_model    |lease_commence_date|resale_price|remaining_lease|
+-------------------+----------+---------+-----+-----------------+------------+--------------+--------------+-------------------+------------+---------------+
|2012-03-01 00:00:00|ANG MO KIO|2 ROOM   |17

In [26]:
print(f"Total Records : {master_df.count()}")
print(f"Total Columns : {len(master_df.columns)}")

Total Records : 89356
Total Columns : 11


<module 'src.config' from '/Users/Kiran/Desktop/HDB-Data-Engineering-Assignment_2026/src/config.py'>

In [35]:
from src.profiler import profile_dataset

profile_dataset(master_df)

Profile report saved to /Users/Kiran/Desktop/HDB-Data-Engineering-Assignment_2026/reports/data_profile.txt


In [38]:
from src.validator import validate_dataset

clean_df, failed_df = validate_dataset(master_df)

In [40]:
print("Master :", master_df.count())
print("Clean  :", clean_df.count())
print("Failed :", failed_df.count())

Master : 89356
Clean  : 89356
Failed : 0


In [41]:
failed_df.show(10, truncate=False)

+-----+----+---------+-----+-----------+------------+--------------+----------+-------------------+------------+---------------+--------------+
|month|town|flat_type|block|street_name|storey_range|floor_area_sqm|flat_model|lease_commence_date|resale_price|remaining_lease|failure_reason|
+-----+----+---------+-----+-----------+------------+--------------+----------+-------------------+------------+---------------+--------------+
+-----+----+---------+-----+-----------+------------+--------------+----------+-------------------+------------+---------------+--------------+



In [42]:
from src.cleaner import recompute_remaining_lease

clean_df = recompute_remaining_lease(clean_df)

In [43]:
clean_df.select(
    "lease_commence_date",
    "remaining_lease"
).show(10, truncate=False)

+-------------------+-----------------+
|lease_commence_date|remaining_lease  |
+-------------------+-----------------+
|1986               |58 Years 6 Months|
|1980               |52 Years 6 Months|
|1980               |52 Years 6 Months|
|1984               |56 Years 6 Months|
|1980               |52 Years 6 Months|
|1981               |53 Years 6 Months|
|1978               |50 Years 6 Months|
|1979               |51 Years 6 Months|
|1979               |51 Years 6 Months|
|1985               |57 Years 6 Months|
+-------------------+-----------------+
only showing top 10 rows


In [70]:
from src.cleaner import remove_duplicate_records

clean_df, duplicate_failed_df = remove_duplicate_records(clean_df)

In [71]:
print("Clean Records :", clean_df.count())
print("Duplicate Records :", duplicate_failed_df.count())

Clean Records : 87796


[Stage 164:============================>                            (1 + 1) / 2]

Duplicate Records : 1560


In [72]:
duplicate_failed_df.show(10, truncate=False)

[Stage 170:============================>                            (1 + 1) / 2]

+-------------------+---------------+---------+-----+-------------------+------------+--------------+----------+-------------------+------------+-----------------+-------------------------------------+
|month              |town           |flat_type|block|street_name        |storey_range|floor_area_sqm|flat_model|lease_commence_date|resale_price|remaining_lease  |failure_reason                       |
+-------------------+---------------+---------+-----+-------------------+------------+--------------+----------+-------------------+------------+-----------------+-------------------------------------+
|2012-03-01 00:00:00|ANG MO KIO     |5 ROOM   |350  |ANG MO KIO ST 32   |11 TO 15    |110.0         |Improved  |2001               |670000.0    |73 Years 6 Months|Duplicate Record - Lower Resale Price|
|2012-03-01 00:00:00|ANG MO KIO     |5 ROOM   |648  |ANG MO KIO AVE 5   |01 TO 05    |121.0         |Improved  |1980               |538000.0    |52 Years 6 Months|Duplicate Record - Lower Resa

In [75]:
from src.validator import detect_price_anomalies

clean_df, anomaly_failed_df = detect_price_anomalies(clean_df)

In [76]:
print("Clean Records    :", clean_df.count())
print("Anomalies Found  :", anomaly_failed_df.count())

Clean Records    : 84671
Anomalies Found  : 3125


In [78]:
from src.transformer import generate_resale_identifier

transformed_df = generate_resale_identifier(clean_df)

In [79]:
transformed_df.select(
    "block",
    "month",
    "town",
    "resale_price",
    "resale_identifier"
).show(10, truncate=False)

[Stage 196:>                                                        (0 + 1) / 1]

+-----+-------------------+----------+------------+-----------------+
|block|month              |town      |resale_price|resale_identifier|
+-----+-------------------+----------+------------+-----------------+
|510  |2012-03-01 00:00:00|ANG MO KIO|265000.0    |S5102503A        |
|172  |2012-03-01 00:00:00|ANG MO KIO|250000.0    |S1722503A        |
|159  |2012-03-01 00:00:00|ANG MO KIO|368000.0    |S1593603A        |
|216  |2012-03-01 00:00:00|ANG MO KIO|368000.0    |S2163603A        |
|313  |2012-03-01 00:00:00|ANG MO KIO|418000.0    |S3133603A        |
|333  |2012-03-01 00:00:00|ANG MO KIO|338000.0    |S3333603A        |
|416  |2012-03-01 00:00:00|ANG MO KIO|434800.0    |S4163603A        |
|418  |2012-03-01 00:00:00|ANG MO KIO|373000.0    |S4183603A        |
|419  |2012-03-01 00:00:00|ANG MO KIO|348000.0    |S4193603A        |
|419  |2012-03-01 00:00:00|ANG MO KIO|380000.0    |S4193603A        |
+-----+-------------------+----------+------------+-----------------+
only showing top 10 

In [81]:
from src.hasher import hash_resale_identifier

hashed_df = hash_resale_identifier(transformed_df)

In [82]:
hashed_df.select(
    "resale_identifier",
    "hashed_resale_identifier"
).show(10, truncate=False)

[Stage 197:============================>                            (1 + 1) / 2]

+-----------------+----------------------------------------------------------------+
|resale_identifier|hashed_resale_identifier                                        |
+-----------------+----------------------------------------------------------------+
|S5102503A        |8dc73a947dbc9d1e58ad5c51f36a97ed4f3529cb3a63a3de034338e156327d55|
|S1722503A        |a659c3bc06c1df52708600c4c565fceadb740c28af68cc168c749d142eb59950|
|S1593603A        |f062e59273ffcaa4e53d394d736f86e669ab0aea31eef100bfda211196ffb3db|
|S2163603A        |912e45923472df3652f4537761c0ea7504b8152560ba073f7877ab03b30d6faf|
|S3133603A        |82f0a061049341dc79895560ee718b54b748e0c776e7e2075744512eff47e9a1|
|S3333603A        |89fbead42226230a6d833797527e3dff2e4118d4d8dd51d7480f6b653f289d6f|
|S4163603A        |d269ec23fbb47de2e39528e70d0f585c30faa958c30f052be9b42771e3a1fc88|
|S4183603A        |af4b9f8ffc9a9cf7a7f7c3bc85e1450850fa8beb24d87e2c798cfc84c804e444|
|S4193603A        |7a8c98b44716d70fe46a4021f1eae165073594fb240c68

In [89]:
from src.writer import (
    write_cleaned,
    write_failed,
    write_transformed,
    write_hashed
)

In [90]:
write_cleaned(clean_df)

[Stage 205:>                                                        (0 + 4) / 4]

✓ Cleaned data written to /Users/Kiran/Desktop/HDB-Data-Engineering-Assignment_2026/data/cleaned


In [91]:
failed_df = (
    duplicate_failed_df
    .unionByName(anomaly_failed_df, allowMissingColumns=True)
)

write_failed(failed_df)

[Stage 207:============================>                            (1 + 1) / 2]

✓ Failed data written to /Users/Kiran/Desktop/HDB-Data-Engineering-Assignment_2026/data/failed


In [92]:
write_transformed(transformed_df)

[Stage 216:>                                                        (0 + 4) / 4]

✓ Transformed data written to /Users/Kiran/Desktop/HDB-Data-Engineering-Assignment_2026/data/transformed


In [93]:
write_hashed(hashed_df)

[Stage 222:>                                                        (0 + 4) / 4]

✓ Hashed data written to /Users/Kiran/Desktop/HDB-Data-Engineering-Assignment_2026/data/hashed


In [94]:
from pathlib import Path

for folder in [
    "cleaned",
    "failed",
    "transformed",
    "hashed"
]:
    print(f"\n{folder.upper()}")

    for file in (PROJECT_ROOT / "data" / folder).glob("*"):
        print(file.name)


CLEANED
part-00003-88e2fab5-edaf-44d6-8707-73a82a173ac4-c000.snappy.parquet
._SUCCESS.crc
.part-00000-88e2fab5-edaf-44d6-8707-73a82a173ac4-c000.snappy.parquet.crc
part-00002-88e2fab5-edaf-44d6-8707-73a82a173ac4-c000.snappy.parquet
.part-00003-88e2fab5-edaf-44d6-8707-73a82a173ac4-c000.snappy.parquet.crc
part-00000-88e2fab5-edaf-44d6-8707-73a82a173ac4-c000.snappy.parquet
part-00001-88e2fab5-edaf-44d6-8707-73a82a173ac4-c000.snappy.parquet
_SUCCESS
.part-00001-88e2fab5-edaf-44d6-8707-73a82a173ac4-c000.snappy.parquet.crc
.part-00002-88e2fab5-edaf-44d6-8707-73a82a173ac4-c000.snappy.parquet.crc

FAILED
.part-00000-56ccb4b8-39cc-4489-ad23-15a111ea547d-c000.snappy.parquet.crc
._SUCCESS.crc
.part-00001-56ccb4b8-39cc-4489-ad23-15a111ea547d-c000.snappy.parquet.crc
part-00000-56ccb4b8-39cc-4489-ad23-15a111ea547d-c000.snappy.parquet
part-00001-56ccb4b8-39cc-4489-ad23-15a111ea547d-c000.snappy.parquet
_SUCCESS

TRANSFORMED
.part-00003-704f7ee1-9989-4c11-9ccd-ceed216c9e65-c000.snappy.parquet.crc
.part